# CNN-LSTM phân loại video ngôn ngữ ký hiệu

**BTL Khai phá dữ liệu – HK261.**

Mô hình đọc 16 khung của mỗi video theo thứ tự thời gian: CNN (EfficientNet-B0) biến mỗi khung thành một vector 1.280 chiều, BiLSTM đọc chuỗi vector theo cả hai chiều thời gian, attention gộp chuỗi thành một vector cho cả video, cuối cùng một lớp tuyến tính phân loại 100 lớp.

Notebook gồm hai giai đoạn:

| Giai đoạn | CNN | Dữ liệu huấn luyện | Mục đích |
|---|---|---|---|
| A | đóng băng, dùng đặc trưng của `extract-features` | 5 bản tăng cường có sẵn + tăng cường trên chuỗi đặc trưng | so sánh trực tiếp với SVM trên cùng đặc trưng; thử đầu vào chuyển động |
| B | tinh chỉnh cùng phần LSTM | đọc video, tăng cường mới ở mỗi epoch | để CNN học tập trung vào động tác thay vì bối cảnh của video |

Epoch dừng được chọn theo Macro F1 trên fold 0 (555 video). Kết quả SVM cho thấy fold 0 lạc quan hơn nhiều so với tập test (0,68 so với 0,30), nên mỗi biến thể đều ghi file nộp để đánh giá thêm trên AIChallenge.

In [ ]:
import os, sys, json, time, math, copy, random, warnings, zipfile, unicodedata
from pathlib import Path

warnings.filterwarnings('ignore', category=SyntaxWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
VAL_FOLD = 0
T = 16
N_CLASSES = 100
QUICK = os.environ.get('QUICK', '0') == '1'
PRETRAINED = os.environ.get('PRETRAINED', '1') == '1'
NUM_WORKERS = int(os.environ.get('NUM_WORKERS', min(4, os.cpu_count() or 1)))
TIME_BUDGET_B = float(os.environ.get('TIME_BUDGET_B', '150')) * 60
WANDB_PROJECT = 'dm-sign-language'

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    ROOT = Path('/kaggle')
    SEARCH_ROOTS, OUT = [ROOT / 'input'], ROOT / 'working'
else:
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd().parents[1]))
    SEARCH_ROOTS = [ROOT / 'models' / 'features', ROOT / 'data', ROOT / 'processing']
    OUT = ROOT / 'models' / 'cnn_lstm' / 'output'
if os.environ.get('SEARCH_ROOTS'):
    SEARCH_ROOTS = [Path(p) for p in os.environ['SEARCH_ROOTS'].split(':')]
OUT = Path(os.environ.get('OUT_DIR', OUT))
OUT.mkdir(parents=True, exist_ok=True)


def find_file(name):
    for root in SEARCH_ROOTS:
        hits = sorted(root.rglob(name)) if root.exists() else []
        if hits:
            return hits[0]
    raise FileNotFoundError(f'Không tìm thấy {name} trong {SEARCH_ROOTS}')


def rel(path):
    return path.relative_to(ROOT) if path.is_relative_to(ROOT) else path


FEAT_DIR = find_file('feat_train.npy').parent
DATASET = find_file('label_mapping.pkl').parent
PP_DIR = FEAT_DIR if (FEAT_DIR / 'preprocess.py').exists() else find_file('preprocess.py').parent
sys.path.insert(0, str(PP_DIR))
import preprocess as pp


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'

# Python 3.12+ in thông báo vô hại khi DataLoader đóng tiến trình con; chỉ ẩn đúng thông báo đó.
_default_unraisable = sys.unraisablehook


def _hide_loader_shutdown(info):
    if '_MultiProcessingDataLoaderIter' not in getattr(info.object, '__qualname__', ''):
        _default_unraisable(info)


sys.unraisablehook = _hide_loader_shutdown

print('Đặc trưng:', rel(FEAT_DIR))
print('Video    :', rel(DATASET))
print('Output   :', rel(OUT))
print('Thiết bị :', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '', '| QUICK =', QUICK)

In [ ]:
try:
    import wandb
except ImportError:
    wandb = None
api_key = os.environ.get('WANDB_API_KEY')
if ON_KAGGLE and not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
    except Exception:
        api_key = None
WANDB_MODE = 'online' if api_key else 'offline'
if wandb is not None:
    os.environ['WANDB_DIR'] = str(OUT)
    os.environ['WANDB_SILENT'] = 'true'
    if api_key:
        wandb.login(key=api_key)


def start_run(name, config):
    if wandb is None:
        return None
    return wandb.init(project=WANDB_PROJECT, name=name, job_type='cnn_lstm', mode=WANDB_MODE,
                      config={'model': 'cnn_lstm', 'backbone': 'efficientnet_b0', 'T': T, 'seed': SEED,
                              'val_fold': VAL_FOLD, 'quick': QUICK, **config})


print('wandb:', WANDB_MODE if wandb is not None else 'chưa cài – bỏ qua')

## 1. Đọc dữ liệu

- `manifest_clean.csv`: nhãn, fold và chỉ số khung hợp lệ của 2.772 video train; thứ tự dòng khớp các file đặc trưng.
- `feat_train.npy` (bản gốc) dùng để đánh giá; `feat_train_aug.npy` (5 bản tăng cường mỗi video) dùng để huấn luyện giai đoạn A.
- Đặc trưng được chuẩn hoá theo trung bình và độ lệch chuẩn của từng chiều, tính trên phần train (fold 1–4).
- Trọng số lớp kiểu `balanced` như SVM: $w_c = N / (K \cdot n_c)$.

In [ ]:
manifest = pd.read_csv(FEAT_DIR / 'manifest_clean.csv')
test_manifest = pd.read_csv(FEAT_DIR / 'test_manifest.csv')
label_names = json.load(open(FEAT_DIR / 'label_names.json', encoding='utf-8'))
display_names = manifest.drop_duplicates('label_idx').set_index('label_idx').cls.sort_index().tolist()
public_rows = test_manifest[test_manifest.split == 'public_test'].reset_index(drop=True)
private_rows = test_manifest[test_manifest.split == 'private_test'].reset_index(drop=True)

y = manifest.label_idx.values
fold = manifest.fold.values
all_idx = np.arange(len(manifest))
tr_idx, va_idx = all_idx[fold != VAL_FOLD], all_idx[fold == VAL_FOLD]

feat_train = np.load(FEAT_DIR / 'feat_train.npy')
feat_aug = np.load(FEAT_DIR / 'feat_train_aug.npy')
feat_public = np.load(FEAT_DIR / 'feat_public.npy')
feat_private = np.load(FEAT_DIR / 'feat_private.npy')
assert feat_train.shape == (len(manifest), T, 1280) and feat_aug.shape[0] == len(manifest)
assert len(feat_public) == len(public_rows) and len(feat_private) == len(private_rows)
assert len(label_names) == N_CLASSES and len(display_names) == N_CLASSES

ref = feat_train[tr_idx].astype(np.float32)
FEAT_MEAN, FEAT_STD = ref.mean((0, 1)), ref.std((0, 1)) + 1e-6
del ref
mean_t = torch.tensor(FEAT_MEAN, device=DEVICE)
std_t = torch.tensor(FEAT_STD, device=DEVICE)
clean_gpu = torch.from_numpy(feat_train).to(DEVICE)
aug_gpu = torch.from_numpy(feat_aug).to(DEVICE)
y_gpu = torch.from_numpy(y).to(DEVICE)

print('Train (fold 1–4):', len(tr_idx), '| validation (fold 0):', len(va_idx))
print('Đặc trưng gốc:', feat_train.shape, '| tăng cường:', feat_aug.shape,
      '| public:', feat_public.shape, '| private:', feat_private.shape)

## 2. Mô hình

| Khối | Cấu hình | Đầu ra |
|---|---|---|
| Đầu vào | 16 vector đặc trưng (giai đoạn A) hoặc 16 khung ảnh qua EfficientNet-B0 (giai đoạn B) | B × 16 × 1.280 |
| Chiếu | Linear → 512, LayerNorm, ReLU, Dropout 0,3 | B × 16 × 512 |
| BiLSTM | 2 lớp, 256 đơn vị mỗi chiều, dropout 0,3 giữa hai lớp | B × 16 × 512 |
| Attention | $e_t = v^\top \tanh(W h_t + b)$, $\alpha_t = \mathrm{softmax}_t(e_t)$, $z = \sum_t \alpha_t h_t$ | B × 512 |
| Phân loại | Dropout 0,4, Linear 512 → 100 | B × 100 |

Attention cho mô hình tự chọn khung quan trọng (lúc tay đang làm ký hiệu) thay vì lấy trung bình đều như SVM. Biến thể **chuyển động** nối mỗi vector với hiệu của nó so với khung trước, $[x_t,\; x_t - x_{t-1}]$, nên đầu vào có 2.560 chiều: phần không đổi qua thời gian (nền, quần áo, khuôn mặt) bị triệt tiêu trong nửa sau.

Ở giai đoạn B, EfficientNet-B0 được nối vào trước phần trên. Phần thân đầu (stem và 2 nhóm khối đầu, học cạnh và màu cơ bản) giữ đóng băng; các lớp BatchNorm giữ thống kê ImageNet vì mỗi batch chỉ có 8 video.

In [ ]:
class TemporalHead(nn.Module):
    def __init__(self, in_dim, proj=512, hidden=256, layers=2, drop=0.3, drop_out=0.4):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, proj), nn.LayerNorm(proj), nn.ReLU(), nn.Dropout(drop))
        self.lstm = nn.LSTM(proj, hidden, num_layers=layers, batch_first=True, bidirectional=True,
                            dropout=drop if layers > 1 else 0.0)
        self.att = nn.Sequential(nn.Linear(2 * hidden, 128), nn.Tanh(), nn.Linear(128, 1, bias=False))
        self.cls = nn.Sequential(nn.Dropout(drop_out), nn.Linear(2 * hidden, N_CLASSES))

    def forward(self, x):
        h, _ = self.lstm(self.proj(x))
        alpha = torch.softmax(self.att(h).squeeze(-1).float(), dim=1)
        z = (alpha.unsqueeze(-1) * h.float()).sum(1)
        return self.cls(z)


def standardize(x):
    return (x.float() - mean_t) / std_t


def add_motion(x):
    return torch.cat([x, torch.diff(x, dim=1, prepend=x[:, :1])], dim=-1)


class CNNLSTM(nn.Module):
    def __init__(self, head, freeze_blocks=3):
        super().__init__()
        cnn = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        cnn.classifier = nn.Identity()
        for p in cnn.features[:freeze_blocks].parameters():
            p.requires_grad = False
        self.cnn, self.head = cnn, head

    def train(self, mode=True):
        super().train(mode)
        for m in self.cnn.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def forward(self, x):
        b = x.shape[0]
        f = self.cnn(x.flatten(0, 1).contiguous(memory_format=torch.channels_last))
        return self.head(standardize(f.view(b, T, -1)))


def n_params(module, trainable=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable) / 1e6


print(f'Phần đầu (đầu vào 1.280 chiều): {n_params(TemporalHead(1280)):.2f} triệu tham số')
print(f'Phần đầu (đầu vào 2.560 chiều): {n_params(TemporalHead(2560)):.2f} triệu tham số')
_probe = CNNLSTM(TemporalHead(1280))
print(f'Giai đoạn B: {n_params(_probe):.2f} triệu tham số, trong đó {n_params(_probe, True):.2f} triệu được huấn luyện')
del _probe

## 3. Giai đoạn A: CNN đóng băng

Mỗi epoch, mỗi video lấy ngẫu nhiên 1 trong 5 bản tăng cường có sẵn, rồi tăng cường thêm ngay trên chuỗi đặc trưng (rẻ vì không phải chạy lại CNN):

- **Che khung**: đặt 0–3 khung ngẫu nhiên về giá trị trung bình, mô phỏng khung mờ hoặc bị che;
- **Nhiễu Gauss** độ lệch chuẩn 0,1 (trên đặc trưng đã chuẩn hoá);
- **Dropout đầu vào** 0,1.

| Siêu tham số | Giá trị |
|---|---|
| Tối ưu | AdamW, learning rate 1e-3, weight decay 1e-2, batch 64 |
| Lịch learning rate | 3 epoch warmup tuyến tính, sau đó giảm theo cosine |
| Hàm mất mát | cross-entropy có trọng số lớp, label smoothing 0,1 |
| Khác | cắt gradient 1,0; tối đa 60 epoch; dừng sớm sau 10 epoch Macro F1 fold 0 không tăng |

Hai biến thể: **A0** (đầu vào gốc) và **A2** (đầu vào chuyển động). Mỗi biến thể huấn luyện trên fold 1–4 để chọn epoch, rồi huấn luyện lại trên cả 2.772 video đúng số epoch đó để ghi file nộp (như SVM).

File nộp có cột `video_name,label`, nhãn lấy từ `label_names.json` (dạng Unicode NFD). AIChallenge chấm trên tập **private test** và đọc file `cv_submission.csv` trong file zip, nên dự đoán private được đóng gói sẵn thành `cnn_lstm_<biến thể>_submission.zip`. **Kiểm tra phụ:** 156 video public test giống hệt video train, nên mô hình phải đoán đúng gần hết; nếu không thì có lỗi lệch thứ tự dòng.

In [ ]:
CFG_A = dict(lr=1e-3, wd=1e-2, batch=64, max_epochs=60, warmup=3, patience=10, smoothing=0.1, clip=1.0,
             mask_max=3, noise=0.1, in_drop=0.1)
if QUICK:
    CFG_A.update(max_epochs=3, patience=2)


def lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))
    return f


def augment_seq(x, cfg):
    b = x.shape[0]
    n_mask = torch.randint(0, cfg['mask_max'] + 1, (b, 1), device=x.device)
    rank = torch.rand(b, T, device=x.device).argsort(1).argsort(1)
    x = x.masked_fill((rank < n_mask).unsqueeze(-1), 0.0)
    x = x + cfg['noise'] * torch.randn_like(x)
    return F.dropout(x, cfg['in_drop'], training=True)


@torch.no_grad()
def predict_feats(model, feats, motion, batch=512):
    model.eval()
    out = []
    for i in range(0, len(feats), batch):
        x = standardize(torch.as_tensor(feats[i:i + batch], device=DEVICE))
        out.append(torch.softmax(model(add_motion(x) if motion else x).float(), -1).cpu())
    return torch.cat(out).numpy()


def evaluate(prob, labels):
    pred = prob.argmax(1)
    nll = float(-np.log(prob[np.arange(len(labels)), labels] + 1e-9).mean())
    return {'val_loss': nll, 'val_f1': f1_score(labels, pred, average='macro'),
            'val_acc': accuracy_score(labels, pred)}


def log(run, row, prefix=''):
    if run is not None:
        run.log({prefix + k: v for k, v in row.items() if k != 'epoch'}, step=row['epoch'] if not prefix else None)


def train_A(motion, train_idx, val_idx=None, epochs=None, run=None, prefix=''):
    seed_everything(SEED)
    cfg = CFG_A
    model = TemporalHead(1280 * (2 if motion else 1)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    crit = nn.CrossEntropyLoss(weight=torch.tensor(pp.class_weights(y[train_idx]), device=DEVICE),
                               label_smoothing=cfg['smoothing'])
    n_epochs = epochs or cfg['max_epochs']
    steps = math.ceil(len(train_idx) / cfg['batch'])
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda(n_epochs * steps, cfg['warmup'] * steps))
    rng = np.random.default_rng(SEED)
    history, best, bad = [], {'val_f1': -1.0, 'epoch': 0, 'state': None}, 0
    for ep in range(1, n_epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        perm = rng.permutation(train_idx)
        for i in range(0, len(perm), cfg['batch']):
            idx = torch.from_numpy(perm[i:i + cfg['batch']]).to(DEVICE)
            k = torch.from_numpy(rng.integers(0, aug_gpu.shape[1], len(idx))).to(DEVICE)
            x = standardize(aug_gpu[idx, k])
            x = augment_seq(add_motion(x) if motion else x, cfg)
            loss = crit(model(x), y_gpu[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            opt.step()
            sched.step()
            total += loss.item() * len(idx)
        row = {'epoch': ep, 'train_loss': total / len(perm), 'lr': opt.param_groups[0]['lr'],
               'seconds': time.time() - t0}
        if val_idx is not None:
            row.update(evaluate(predict_feats(model, feat_train[val_idx], motion), y[val_idx]))
            if row['val_f1'] > best['val_f1']:
                best = {'val_f1': row['val_f1'], 'epoch': ep, 'state': copy.deepcopy(model.state_dict())}
                bad = 0
            else:
                bad += 1
        history.append(row)
        log(run, row, prefix)
        if val_idx is not None and (ep % 5 == 0 or ep == 1):
            print(f"  epoch {ep:2d} | train loss {row['train_loss']:.3f} | val loss {row['val_loss']:.3f} "
                  f"| val F1 {row['val_f1']:.4f} | val acc {row['val_acc']:.4f}")
        if val_idx is not None and bad >= cfg['patience']:
            break
    if best['state'] is not None:
        model.load_state_dict(best['state'])
    return model, pd.DataFrame(history), best['epoch']

In [ ]:
md5_label = dict(zip(manifest.md5, manifest.label_idx))
dup_mask = public_rows.md5.map(md5_label).notna().values
dup_label = public_rows.md5.map(md5_label)[dup_mask].astype(int).values


def write_submission(name, prob_private, prob_public):
    info = {}
    for split, rows, prob in [('private', private_rows, prob_private), ('public', public_rows, prob_public)]:
        rows = rows.iloc[:len(prob)]
        pred = prob.argmax(1)
        sub = pd.DataFrame({'video_name': rows.file.values, 'label': [label_names[i] for i in pred]})
        if not QUICK:
            assert len(sub) == {'private': 2859, 'public': 1630}[split]
        assert sub.video_name.is_unique and set(sub.label) <= set(label_names)
        assert all(unicodedata.normalize('NFD', s) == s for s in sub.label)
        sub.to_csv(OUT / f'cnn_lstm_{name}_{split}.csv', index=False, encoding='utf-8')
        info[f'n_classes_{split}'] = int(sub.label.nunique())
    with zipfile.ZipFile(OUT / f'cnn_lstm_{name}_submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(OUT / f'cnn_lstm_{name}_private.csv', arcname='cv_submission.csv')
    mask = dup_mask[:len(prob_public)]
    if mask.sum():
        info['public_dup_agree'] = float((prob_public.argmax(1)[mask] == dup_label[:mask.sum()]).mean())
    return info


def finish_run(run, r):
    if run is not None:
        run.summary.update({'val_macro_f1': round(float(r['fold0_f1']), 4), 'val_accuracy': round(float(r['fold0_acc']), 4),
                            'best_epoch': int(r['best_epoch']), 'n_classes_private': r['n_classes_private']})
        run.finish()

In [ ]:
VARIANTS_A = {'A0': {'motion': False}, 'A2': {'motion': True}}
results, histories = {}, []
for name, v in VARIANTS_A.items():
    run = start_run(f'cnnlstm-{name}', {'stage': 'A', 'variant': name, **v, **CFG_A})
    print(f'{name} ({"chuyển động" if v["motion"] else "đầu vào gốc"}): huấn luyện trên fold 1–4')
    t0 = time.time()
    model, hist, best_ep = train_A(v['motion'], tr_idx, va_idx, run=run)
    prob_val = predict_feats(model, feat_train[va_idx], v['motion'])
    m = evaluate(prob_val, y[va_idx])
    r = {'model_sel': model, 'motion': v['motion'], 'best_epoch': best_ep, 'prob_val': prob_val,
         'fold0_f1': m['val_f1'], 'fold0_acc': m['val_acc']}
    histories.append(hist.assign(run=name))
    print(f'  -> epoch tốt nhất {best_ep}, fold 0: Macro F1 {m["val_f1"]:.4f}, accuracy {m["val_acc"]:.4f} '
          f'({(time.time() - t0) / 60:.1f} phút)')

    t0 = time.time()
    final, _, _ = train_A(v['motion'], all_idx, epochs=best_ep, run=run, prefix='refit_')
    r.update(write_submission(name, predict_feats(final, feat_private, v['motion']),
                              predict_feats(final, feat_public, v['motion'])))
    torch.save({'state_dict': final.state_dict(), 'motion': v['motion'], 'epochs': best_ep,
                'feat_mean': FEAT_MEAN, 'feat_std': FEAT_STD, 'label_names': label_names},
               OUT / f'cnn_lstm_{name}.pth')
    print(f"  -> huấn luyện lại trên {len(all_idx)} video, {best_ep} epoch, {(time.time() - t0) / 60:.1f} phút | "
          f"private: {r['n_classes_private']} lớp được dự đoán | public trùng train đoán đúng "
          f"{r.get('public_dup_agree', float('nan')):.1%} | file nộp cnn_lstm_{name}_submission.zip\n")
    results[name] = r
    finish_run(run, r)

## 4. Giai đoạn B: tinh chỉnh EfficientNet-B0 cùng phần LSTM

Giai đoạn A chỉ học phần sau CNN, nên CNN vẫn mô tả cả khung hình theo cách đã học từ ImageNet (nền, quần áo, khuôn mặt cũng được mã hoá). Giai đoạn B cho CNN học tiếp trên video ký hiệu, với tăng cường mới ở mỗi epoch, để đặc trưng tập trung vào bàn tay và chuyển động.

- **Khởi tạo:** CNN từ ImageNet, phần LSTM từ mô hình A0 đã huấn luyện trên fold 1–4, nên mô hình bắt đầu từ mức của A0.
- **Tăng cường video** (`preprocess.VideoAugmentation`): giữ ngẫu nhiên 80–100% đoạn thời gian, cắt ngẫu nhiên 75–100% khung hình, đổi độ sáng, độ tương phản, độ bão hoà ±25%. Không lật ngang vì tay trái và tay phải có thể đổi nghĩa ký hiệu.

| Siêu tham số | Giá trị |
|---|---|
| Learning rate | 1e-4 cho CNN, 5e-4 cho phần LSTM; 1 epoch warmup rồi cosine |
| Batch | 8 video (128 khung), mixed precision trên GPU |
| Epoch | tối đa 15, dừng sớm sau 4 epoch Macro F1 fold 0 không tăng |
| Khác | weight decay 1e-2, label smoothing 0,1, trọng số lớp, cắt gradient 1,0 |

Mỗi epoch đọc lại toàn bộ video nên tốn thời gian hơn nhiều so với giai đoạn A; mô hình nộp bài là checkpoint có Macro F1 fold 0 cao nhất (không huấn luyện lại trên cả train).

In [ ]:
CFG_B = dict(lr_cnn=1e-4, lr_head=5e-4, wd=1e-2, batch=8, max_epochs=15, warmup=1, patience=4, smoothing=0.1,
             clip=1.0, freeze_blocks=3, crop_min=0.75)
if QUICK:
    CFG_B.update(max_epochs=1, batch=1)


class VideoClips(Dataset):
    def __init__(self, paths, valid=None, labels=None, aug=None):
        self.paths, self.valid, self.labels, self.aug = list(paths), valid, labels, aug
        self.rng = None

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        if self.aug is not None and self.rng is None:
            self.rng = np.random.default_rng(torch.initial_seed() % 2 ** 32)
        v = self.valid[i] if self.valid is not None else None
        clip = pp.load_clip(self.paths[i], valid_idx=v, aug=self.aug, rng=self.rng)
        return torch.from_numpy(pp.normalize(clip)), int(self.labels[i]) if self.labels is not None else -1


def loader(ds, batch, shuffle=False):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, drop_last=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0, generator=g)


dirs = pp.class_dirs(DATASET / 'train')
train_paths = np.array([DATASET / 'train' / dirs[pp.nfc(c)] / f for c, f in zip(manifest.cls, manifest.file)])
valid_idx = manifest.valid_idx.values
b_tr, b_va = tr_idx, va_idx
b_public, b_private = public_rows, private_rows
if QUICK:
    b_tr, b_va, b_public, b_private = tr_idx[:16], va_idx[:8], public_rows.head(8), private_rows.head(8)

AUG_B = pp.VideoAugmentation(crop_scale=(CFG_B['crop_min'], 1.0))
train_loader = loader(VideoClips(train_paths[b_tr], valid_idx[b_tr], y[b_tr], AUG_B), CFG_B['batch'], shuffle=True)
val_loader = loader(VideoClips(train_paths[b_va], valid_idx[b_va], y[b_va]), 2 * CFG_B['batch'])
print('Giai đoạn B: train', len(b_tr), 'video,', len(train_loader), 'batch mỗi epoch | validation', len(b_va), 'video')


@torch.no_grad()
def predict_videos(model, dl):
    model.eval()
    out = []
    for x, _ in dl:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            out.append(torch.softmax(model(x.to(DEVICE, non_blocking=True)).float(), -1).cpu())
    return torch.cat(out).numpy()

In [ ]:
del aug_gpu, feat_aug
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
seed_everything(SEED)
model_B = CNNLSTM(copy.deepcopy(results['A0']['model_sel']), CFG_B['freeze_blocks']).to(DEVICE)
model_B = model_B.to(memory_format=torch.channels_last)
cnn_params = [p for p in model_B.cnn.parameters() if p.requires_grad]
opt = torch.optim.AdamW([{'params': cnn_params, 'lr': CFG_B['lr_cnn']},
                         {'params': model_B.head.parameters(), 'lr': CFG_B['lr_head']}], weight_decay=CFG_B['wd'])
steps = len(train_loader)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda(CFG_B['max_epochs'] * steps, CFG_B['warmup'] * steps))
crit = nn.CrossEntropyLoss(weight=torch.tensor(pp.class_weights(y[b_tr]), device=DEVICE),
                           label_smoothing=CFG_B['smoothing'])
scaler = torch.amp.GradScaler('cuda', enabled=AMP)
run_B = start_run('cnnlstm-B', {'stage': 'B', 'variant': 'B', 'init_head': 'A0', **CFG_B})

history_B, best_B, bad, t_start = [], {'val_f1': -1.0, 'epoch': 0, 'state': None}, 0, time.time()
for ep in range(1, CFG_B['max_epochs'] + 1):
    model_B.train()
    t0, total, seen = time.time(), 0.0, 0
    for x, yb in train_loader:
        x, yb = x.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            logits = model_B(x)
        loss = crit(logits.float(), yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model_B.parameters(), CFG_B['clip'])
        scaler.step(opt)
        scaler.update()
        sched.step()
        total, seen = total + loss.item() * len(yb), seen + len(yb)
    row = {'epoch': ep, 'train_loss': total / seen, 'lr': opt.param_groups[0]['lr'], 'seconds': time.time() - t0}
    row.update(evaluate(predict_videos(model_B, val_loader), y[b_va]))
    if row['val_f1'] > best_B['val_f1']:
        best_B = {'val_f1': row['val_f1'], 'epoch': ep,
                  'state': {k: v.detach().cpu().clone() for k, v in model_B.state_dict().items()}}
        bad = 0
    else:
        bad += 1
    history_B.append(row)
    log(run_B, row)
    print(f"epoch {ep:2d} | train loss {row['train_loss']:.3f} | val loss {row['val_loss']:.3f} | "
          f"val F1 {row['val_f1']:.4f} | val acc {row['val_acc']:.4f} | {row['seconds'] / 60:.1f} phút")
    elapsed = time.time() - t_start
    if bad >= CFG_B['patience']:
        print(f'Dừng sớm: {bad} epoch liền Macro F1 fold 0 không tăng.')
        break
    if elapsed + elapsed / ep > TIME_BUDGET_B:
        print(f'Dừng vì giới hạn thời gian ({TIME_BUDGET_B / 60:.0f} phút).')
        break

model_B.load_state_dict(best_B['state'])
print(f"Epoch tốt nhất {best_B['epoch']}: Macro F1 fold 0 = {best_B['val_f1']:.4f} | "
      f"tổng {(time.time() - t_start) / 60:.1f} phút")

In [ ]:
t0 = time.time()
prob_val_B = predict_videos(model_B, val_loader)
prob_public_B = predict_videos(model_B, loader(VideoClips([DATASET / 'public_test' / f for f in b_public.file]), 2 * CFG_B['batch']))
prob_private_B = predict_videos(model_B, loader(VideoClips([DATASET / 'private_test' / f for f in b_private.file]), 2 * CFG_B['batch']))
m = evaluate(prob_val_B, y[b_va])
results['B'] = {'motion': False, 'best_epoch': best_B['epoch'], 'prob_val': prob_val_B,
                'fold0_f1': m['val_f1'], 'fold0_acc': m['val_acc'],
                **write_submission('B', prob_private_B, prob_public_B)}
histories.append(pd.DataFrame(history_B).assign(run='B'))
torch.save({'state_dict': best_B['state'], 'epoch': best_B['epoch'], 'config': CFG_B,
            'feat_mean': FEAT_MEAN, 'feat_std': FEAT_STD, 'label_names': label_names}, OUT / 'cnn_lstm_B.pth')
r = results['B']
print(f"Dự đoán test trong {(time.time() - t0) / 60:.1f} phút | private: {r['n_classes_private']} lớp được dự đoán | "
      f"public trùng train đoán đúng {r.get('public_dup_agree', float('nan')):.1%} | file nộp cnn_lstm_B_submission.zip")

## 5. So sánh các biến thể

Bảng dưới tóm tắt kết quả trên fold 0 (chọn epoch theo fold 0 nên con số hơi lạc quan). Hình: Macro F1 fold 0 và loss theo epoch; giai đoạn B bắt đầu từ mô hình A0 nên trục epoch của B là số epoch tinh chỉnh thêm.

In [ ]:
history = pd.concat(histories, ignore_index=True)
history.to_csv(OUT / 'history.csv', index=False)
table = pd.DataFrame([{'biến thể': k, 'epoch tốt nhất': r['best_epoch'], 'fold 0 Macro F1': round(r['fold0_f1'], 4),
                       'fold 0 accuracy': round(r['fold0_acc'], 4), 'số lớp dự đoán (private)': r['n_classes_private']}
                      for k, r in results.items()])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
colors = {'A0': '#94a3b8', 'A2': '#f59e0b', 'B': '#2563eb'}
for name, h in history.groupby('run', sort=False):
    axes[0].plot(h.epoch, h.val_f1, 'o-', ms=3, color=colors[name], label=name)
    axes[1].plot(h.epoch, h.train_loss, '-', color=colors[name], label=f'{name} train')
    axes[1].plot(h.epoch, h.val_loss, '--', color=colors[name], label=f'{name} val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('Macro F1 (fold 0)'); axes[0].set_title('Macro F1 trên fold 0')
axes[0].legend()
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss'); axes[1].set_title('Loss (liền: train, đứt: fold 0)')
axes[1].legend(fontsize=7, ncol=3)
fig.tight_layout(); fig.savefig(OUT / 'cnn_lstm_curves.png', dpi=150, bbox_inches='tight'); plt.show()
table

## 6. Phân tích mô hình giai đoạn B trên fold 0

In [ ]:
pred_val = results['B']['prob_val'].argmax(1)
y_val = y[b_va]
rep = classification_report(y_val, pred_val, labels=range(N_CLASSES), target_names=display_names,
                            output_dict=True, zero_division=0)
per_class = (pd.DataFrame(rep).T.iloc[:N_CLASSES].rename_axis('class').reset_index()
             .assign(label_idx=range(N_CLASSES)))[['label_idx', 'class', 'precision', 'recall', 'f1-score', 'support']]
per_class.to_csv(OUT / 'fold0_report_B.csv', index=False)
print('10 lớp có F1 thấp nhất trên fold 0 (mỗi lớp chỉ 1–10 video nên chỉ để tham khảo):')
print(per_class.sort_values(['f1-score', 'support']).head(10).to_string(index=False))

cm = confusion_matrix(y_val, pred_val, labels=range(N_CLASSES))
np.fill_diagonal(cm, 0)
pairs = sorted([(display_names[i], display_names[j], cm[i, j]) for i, j in zip(*np.nonzero(cm))], key=lambda t: -t[2])
print('\nCác cặp hay nhầm nhất (nhãn thật -> dự đoán, số video):')
for a, b, n in pairs[:10]:
    print(f'  {a} -> {b}: {n}')

fig, ax = plt.subplots(figsize=(11, 3.2))
s = per_class.sort_values('f1-score', ascending=False)
ax.bar(range(N_CLASSES), s['f1-score'], color='#2563eb', width=0.8)
ax.axhline(results['B']['fold0_f1'], ls='--', c='k', lw=1)
ax.text(99, results['B']['fold0_f1'] + 0.02, f"Macro F1 = {results['B']['fold0_f1']:.3f}", ha='right', fontsize=9)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(s['class'], rotation=90, fontsize=5.5)
ax.set_ylabel('F1 trên fold 0'); ax.set_ylim(0, 1.05); ax.set_title('CNN-LSTM giai đoạn B: F1 từng lớp trên fold 0')
fig.tight_layout(); fig.savefig(OUT / 'cnn_lstm_B_fold0_per_class_f1.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Phân bố dự đoán trên tập private

Với SVM, dự đoán trên private dồn vào một số ít lớp (lớp nhiều nhất 147 video, 9 lớp không được đoán lần nào), dấu hiệu video test khác video train. Bảng dưới kiểm tra điều tương tự cho từng biến thể; nếu chia đều thì mỗi lớp khoảng 29 video.

In [ ]:
rows = []
for name in results:
    pred = pd.read_csv(OUT / f'cnn_lstm_{name}_private.csv').label
    counts = pred.value_counts()
    rows.append({'biến thể': name, 'số lớp được dự đoán': counts.size, 'lớp nhiều nhất': unicodedata.normalize('NFC', counts.index[0]),
                 'số video của lớp đó': int(counts.iloc[0]), '10 lớp nhiều nhất chiếm': f'{counts.head(10).sum() / len(pred):.0%}'})
pd.DataFrame(rows)

## 8. Lưu kết quả và log

In [ ]:
summary = {'fold0': {k: {'best_epoch': int(r['best_epoch']), 'macro_f1': round(float(r['fold0_f1']), 4),
                         'accuracy': round(float(r['fold0_acc']), 4), 'n_classes_private': r['n_classes_private']}
                     for k, r in results.items()},
           'config_A': CFG_A, 'config_B': CFG_B, 'quick': QUICK, 'device': str(DEVICE), 'torch': torch.__version__}
json.dump(summary, open(OUT / 'summary.json', 'w'), indent=2, ensure_ascii=False)

if run_B is not None:
    run_B.log({'fold0_per_class': wandb.Table(dataframe=per_class),
               'curves': wandb.Image(str(OUT / 'cnn_lstm_curves.png')),
               'fold0_per_class_plot': wandb.Image(str(OUT / 'cnn_lstm_B_fold0_per_class_f1.png'))})
    finish_run(run_B, results['B'])

print(json.dumps(summary['fold0'], indent=2, ensure_ascii=False))
for p in sorted(q for q in OUT.iterdir() if not q.name.startswith('.')):
    print(f'{p.name:36s}', 'thư mục' if p.is_dir() else f'{p.stat().st_size / 1e6:.2f} MB')